# VYTRIX: Intelligent Spatial Engine & Photorealistic 3D Digital Twin

**Team:** VYTRO SQUAD  
**Competition:** Techfest 2026-27, IIT Bombay (Round 1)  
**Repository:** [https://github.com/coder-mrjk/vytrix.git](https://github.com/coder-mrjk/vytrix.git)  

---

## Colab Execution Guide for Round 1
This official driver notebook executes the complete **VYTRIX 6-Stage Reconstruction Pipeline** on a free **Google Colab T4 GPU**:

1. **Stage 1: Preprocess** (Adaptive decimation & Laplacian blur filtering to ~350 frames)
2. **Stage 2: Structure-from-Motion** (COLMAP feature extraction & auto matcher with CPU fallback)
3. **Stage 3: 3D Gaussian Splatting** (`gsplat==1.4.0` `TODO(verify)` rasterizer training with Google Drive checkpoints)
4. **Stage 4: Canonical Normalization** (Camera up-vector aligned floor RANSAC, $+Y$ up, $Y=0$, metric scale)
5. **Stage 5: Optimization & Pruning** (Opacity pruning & standard 32-byte binary `.splat` packing)
6. **Stage 6: Packaging** (Strict evidence-based `scene.json` manifest & web viewer bundle)

> **Runtime Requirement:** Ensure this notebook is connected to a GPU runtime (**Runtime $\rightarrow$ Change runtime type $\rightarrow$ T4 GPU**).

### Step 1: GPU & Hardware Verification
**Expected Duration:** ~5 seconds (illustrative benchmark example)  
**Action:** Verify that an NVIDIA CUDA-enabled GPU (T4, V100, or A100) is active.

In [ ]:
!nvidia-smi

import torch

assert torch.cuda.is_available(), "ERROR: CUDA GPU is not available! Change runtime type to T4 GPU."
print(f"CUDA Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Step 2: Install System & Python Dependencies
**Expected Duration:** ~2–4 minutes (illustrative benchmark example)  
**Details:**
- Installs COLMAP, FFmpeg, and Ninja compiler via `apt`
- Installs pinned Python packages: `pyyaml==6.0.1`, `numpy>=1.24.0,<2.0.0`, `scipy>=1.11.0`, `opencv-python-headless>=4.8.0`, `pytest>=7.4.0`
- Installs `gsplat==1.4.0` `TODO(verify)` (Apache-2.0 license, lean footprint, fits free T4 GPU VRAM)
- Clones `gsplat` v1.4.0 source for `examples/simple_trainer.py` and installs required trainer dependencies (including `pycolmap`)
- *Note on CUDA compilation:* On first training invocation, PyTorch/Ninja JIT-compiles CUDA rasterization kernels (takes ~1–2 minutes).

In [ ]:
# 1. Install COLMAP and FFmpeg
!apt-get update -qq && apt-get install -y -qq colmap ffmpeg ninja-build

# 2. Install pinned core Python packages
!pip install -q pyyaml==6.0.1 "numpy>=1.24.0,<2.0.0" "scipy>=1.11.0" "opencv-python-headless>=4.8.0" "pytest>=7.4.0"

# 3. Install gsplat and clone trainer source
# TODO(verify): Validate gsplat release on live Colab run
!pip install -q gsplat==1.4.0

!if [ ! -d "/content/gsplat_src" ]; then \
    git clone --depth 1 --branch v1.4.0 https://github.com/nerfstudio-project/gsplat.git /content/gsplat_src; \
 fi

# 4. Install trainer dependencies (including pycolmap)
!pip install -q -r /content/gsplat_src/examples/requirements.txt

print("All VYTRIX system and training dependencies installed successfully!")

### Step 3: Mount Google Drive (Persistent Checkpoints)
**Expected Duration:** ~20 seconds (illustrative benchmark example)  
**Action:** Mounts Google Drive so training checkpoints and the final `.splat` model are safely preserved if Colab disconnects.

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')
checkpoint_dir = '/content/drive/MyDrive/VYTRIX_Checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)
print(f"Google Drive checkpoint directory ready: {checkpoint_dir}")

### Step 4: Clone the VYTRIX Codebase & Run Unit Tests
**Expected Duration:** ~20 seconds (illustrative benchmark example)  
**Action:** Clones latest code from GitHub into `/content/vytrix` and runs the full unit test suite (15 tests).

In [ ]:
%cd /content
!if [ -d "vytrix" ]; then rm -rf vytrix; fi
!git clone https://github.com/coder-mrjk/vytrix.git
%cd /content/vytrix

# Run unit test suite to verify math and pipeline logic
!PYTHONPATH=pipeline pytest test/ -v

### Step 5: Dry-Run Pipeline Validation (Smoke Test)
**Expected Duration:** ~3–5 minutes (illustrative benchmark example on a Colab T4 instance)  
**Purpose:** Runs a rapid end-to-end smoke test of all 6 stages on a short 30–60 second smartphone clip (extracting 30 frames and running 200 training steps). This verifies COLMAP feature extraction, PyTorch/Ninja CUDA JIT compilation, RANSAC floor leveling, and 32-byte `.splat` packing before you spend 40+ minutes on your full capture.

> **Important:** COLMAP Structure-from-Motion requires real multiview 3D parallax and cannot reconstruct synthetic 2D frames. You must pass a real 30–60s video clip via `--video`.

**Illustrative Benchmark Example Output to Confirm:**
```text
[PASS] Stage 1 (Preprocess): Extracted 30 frames (1.8s)
[PASS] Stage 2 (COLMAP SfM): 28/30 images registered (24.2s)
[PASS] Stage 3 (3DGS Training): 200 steps & CUDA compilation passed (48.1s)
[PASS] Stage 4 (Normalization): Floor leveled to Y=0 (0.4s)
[PASS] Stage 5 (Optimize): 32-byte .splat binary created (0.2s)
[PASS] Stage 6 (Packaging): Manifest generated (0.1s)
```
*(Note: Timings are illustrative benchmarks from a reference T4 run. Each stage evaluates strictly measured conditions and fails if files, registrations, or inliers are below thresholds).* If any stage prints `[FAIL]`, copy the error and send it back to Antigravity.

In [ ]:
# Upload a short 30-60 second smartphone test clip for the smoke test
import os

from google.colab import files

print("Upload a short 30-60s smartphone clip for the smoke test:")
uploaded_smoke = files.upload()
smoke_clip_path = os.path.abspath(next(iter(uploaded_smoke.keys())))

# Execute fast dry-run smoke test
!PYTHONPATH=pipeline python -m vytrix.dry_run \
    --video "$smoke_clip_path" \
    --work-dir "data/dry_run_work"

### Step 6: Full Capture Video Ingestion
**Expected Duration:** Dependent on upload speed (~1–3 minutes)  
**Action:** Upload your complete home walkthrough MP4 (recorded following `docs/SOP.md`: slow continuous loop, 1080p/4K, locked exposure/focus, ~1–2 minutes).

In [ ]:
import os

from google.colab import files

print("Please select your complete smartphone capture video (MP4)...")
uploaded = files.upload()
video_name = next(iter(uploaded.keys()))
video_path = os.path.abspath(video_name)
print(f"Uploaded capture video: {video_path}")

### Step 7: Execute Full Reconstruction & Two-Pass Scale Calibration

#### The Exact Two-Pass Procedure:

**Pass 1: Full Uncalibrated Reconstruction (~30–50 min on Colab T4)**
- Runs all 6 stages uncalibrated from the uploaded capture video.
- Produces `data/work/train/splat_raw.ply` and packages the initial uncalibrated `scene.json` and `scene.splat`.
- Download the bundle (Step 8), open in the local web viewer, and press **`P`** to enter **Scale Calibration Mode**.
- Aim at Point 1 on your floor tape (or reference object) and click; aim at Point 2 and click. Enter the real measured distance in metres (e.g. `1.00`) and click **COPY CLI ARGUMENTS**.

**Pass 2: Fast Metric Recalibration without Retraining (~15–20 seconds)**
- Paste your copied `--scale-points "x1,y1,z1,x2,y2,z2"` and `--scale-distance 1.00` below.
- Running with `--from-stage 4` forces Stages 4, 5, and 6 to execute while **skipping Stages 1, 2, and 3**.
- **Stage 3 output (`splat_raw.ply`) is reused directly without any retraining**!
- Stage 4 scales the Gaussians and adjusts canonical bounds; Stage 5 converts to `.splat`; Stage 6 updates `scene.json` with `scale_method: "two_point_tape"`.

In [ ]:
import sys

sys.path.insert(0, "pipeline")

# =====================================================================
# PASS 1: Set RUN_PASS_2 = False to perform full reconstruction.
# PASS 2: After picking points in viewer, set RUN_PASS_2 = True,
#         paste SCALE_POINTS, and rerun this cell (takes ~15s, no retraining).
# =====================================================================
RUN_PASS_2 = False
SCALE_POINTS = None        # e.g. "-0.45,0.0,0.12,0.55,0.0,0.12"
SCALE_DISTANCE = 1.00      # Real-world distance in metres

if not RUN_PASS_2:
    print("--- EXECUTING PASS 1: Full Uncalibrated Reconstruction ---")
    !PYTHONPATH=pipeline python -m vytrix.run \
        --video "$video_path" \
        --config "config.yaml" \
        --work-dir "data/work" \
        --force
else:
    print("--- EXECUTING PASS 2: Recalibrating Stages 4-6 (Reusing Stage 3 Trained PLY) ---")
    assert SCALE_POINTS, "ERROR: SCALE_POINTS must be set for Pass 2!"
    !PYTHONPATH=pipeline python -m vytrix.run \
        --config "config.yaml" \
        --work-dir "data/work" \
        --from-stage 4 \
        --scale-points "$SCALE_POINTS" \
        --scale-distance $SCALE_DISTANCE

### Step 8: Inspect Real Metrics & Download Web Viewer Bundle
**Action:** Verify that all metrics in `scene.json` are real, zip the web scene bundle, and download to your local computer.

**Logs to Copy & Send Back to Antigravity:**
1. Number of registered images and mean reprojection error (from Stage 2)
2. Final training step and PSNR/SSIM or loss (from Stage 3)
3. Floor plane equation and inlier count (from Stage 4)
4. Final Gaussian count and file size in MB of `scene.splat` (from Stage 5)

In [ ]:
import json
import shutil

from google.colab import files

# 1. Display real scene manifest metrics
manifest_path = "viewer/public/scenes/default/scene.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)

print("=================== VYTRIX REAL SCENE METRICS ===================")
print(f"Scene ID:               {manifest.get('scene_id')}")
print(f"Title:                  {manifest.get('title')}")
print(f"Registered Images:      {manifest.get('metrics', {}).get('registered_images')} / {manifest.get('metrics', {}).get('total_images')}")
print(f"Mean Reprojection Err:  {manifest.get('metrics', {}).get('reproj_error_px')} px")
print(f"Trained Gaussians:      {manifest.get('splat', {}).get('num_gaussians'):,}")
print(f"Splat File Size:        {manifest.get('splat', {}).get('bytes') / (1024*1024):.2f} MB")
print(f"Scale Method:           {manifest.get('transform_applied', {}).get('scale_method')}")
print(f"Bounding Box:           {manifest.get('bounds')}")
print(f"Spawn Position:         {manifest.get('spawn')}")
print("==================================================================")

# 2. Zip and download bundle
bundle_zip = "vytrix_scene_bundle"
shutil.make_archive(bundle_zip, "zip", "viewer/public/scenes/default")
print(f"Scene bundle ready: {bundle_zip}.zip")
files.download(f"{bundle_zip}.zip")
print("Download initiated! Unzip this into 'viewer/public/scenes/default/' locally to roam your real home.")